# FAISS index — sanity check

Builds the dense index from the precomputed chunk embeddings
(`data/raw/radboud-val.jina-v5-nano.chunks-1024.parquet`) and verifies it is
wired up correctly **before** we compute retrieval metrics.

What we're checking:

1. `load_chunk_embeddings` returns a `(302736, 768)` `float32` matrix with a position-aligned `doc_ids` array (one row per chunk).
2. Every row is unit length after re-normalization, so inner product == cosine.
3. `build_index` returns a populated `IndexFlatIP` with `ntotal == n_chunks` and `d == 768`.
4. **Self-search**: handing a stored vector back to FAISS returns that same vector at rank 1 with score ≈ 1.0 — the geometric "is everything plugged in correctly" check. The matching `doc_id` confirms the row→doc mapping survives the round-trip.

Query vectors are precomputed (`...queries.parquet`), so the next step after this is the metrics.

In [ ]:
import numpy as np
from ows_retrieval.data import load_chunk_embeddings
from ows_retrieval.dense import build_index

# One row per chunk; doc_ids is position-aligned with chunk_vectors.
chunk_vectors, doc_ids = load_chunk_embeddings()

print('chunk_vectors shape:', chunk_vectors.shape)
print('chunk_vectors dtype:', chunk_vectors.dtype)
print('doc_ids shape      :', doc_ids.shape)
print('lengths agree      :', len(chunk_vectors) == len(doc_ids))
print('distinct docs      :', len(np.unique(doc_ids)))

## 1. Raw row norms

The stored vectors are already L2-normalized, but float32 rounding leaves the norms at `1.0 ± ~4e-3`. We check that here (expect everything within ~1e-2 of 1.0), then tighten them to exact unit length in the next step.

In [ ]:
raw_norms = np.linalg.norm(chunk_vectors, axis=1)
print('raw min norm :', raw_norms.min())
print('raw max norm :', raw_norms.max())
print('raw mean norm:', raw_norms.mean())
print('all within 1e-2 of 1.0:', np.allclose(raw_norms, 1.0, atol=1e-2))

## 2. Re-normalize for exact cosine

Dividing each row by its exact norm makes inner product == cosine to float precision. We also guard against **zero-norm rows** — a zero row would divide to NaN and poison every score it touches. We expect none here, so no filtering is needed.

In [ ]:
n_zero = int((raw_norms == 0).sum())
print('zero-norm rows:', n_zero, '(expected 0)')

# No zero-norm rows, so the division is safe (no NaNs introduced).
chunk_vectors = chunk_vectors / raw_norms[:, None]

norms = np.linalg.norm(chunk_vectors, axis=1)
print('post-norm min :', norms.min())
print('post-norm max :', norms.max())
print('all within 1e-6 of 1.0:', np.allclose(norms, 1.0, atol=1e-6))
print('any NaN rows  :', bool(np.isnan(chunk_vectors).any()))

## 3. Build the index

In [ ]:
index = build_index(chunk_vectors)

print('index.ntotal:', index.ntotal, '(expected', len(chunk_vectors), ')')
print('index.d     :', index.d, '(expected 768)')
print('match       :', index.ntotal == len(chunk_vectors) and index.d == 768)

## 4. Self-search

Pick an arbitrary stored row, hand it back to FAISS as the query, and ask for top-5. The top-1 hit must be that exact row with score ≈ 1.0 (a unit vector dotted with itself), and its `doc_id` must match the probe's — confirming the parallel `doc_ids` mapping survives a round-trip.

(Chunks overlap by 128 tokens, so the runners-up are typically other chunks of the same document — expected, not a bug.)

In [ ]:
probe_idx = 12345  # arbitrary chunk
query = chunk_vectors[probe_idx:probe_idx + 1]  # shape (1, 768)

scores, idxs = index.search(query, 5)
print('top-5 idxs  :', idxs[0].tolist())
print('top-5 scores:', [round(float(s), 4) for s in scores[0]])
print()
print('top-1 idx matches probe :', idxs[0][0] == probe_idx)
print('top-1 score ~ 1.0       :', abs(float(scores[0][0]) - 1.0) < 1e-3)
print('top-1 doc_id            :', doc_ids[idxs[0][0]])
print('expected doc_id         :', doc_ids[probe_idx])

### Conclusion

The index is wired correctly and ready for retrieval metrics:

- **302,736 chunks**, each a unit vector (norms == 1.0 to float precision), no zero-norm or NaN rows — so inner product == cosine.
- `IndexFlatIP` is populated with `ntotal == 302736` and `d == 768`.
- **Self-search round-trips:** a stored vector handed back to FAISS comes back at rank 1 with score ≈ 1.0, and its `doc_id` matches — the parallel `doc_ids` mapping survives the round-trip.

Next step is the actual retrieval evaluation: `dense.search` with the precomputed query vectors, max-over-chunks aggregation per `doc_id`, then nDCG@k / Recall@k against qrels (compared with the BM25 baseline).